In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import torchvision.ops
import pickle
import time
import argparse
import os
import importlib



from utils.tools import prepare_target_Rall, find_not_all_nan_times_new, derive_train_val_idxs_new, derive_train_val_idxs_CORDEX
from utils.tools import derive_qmse_bins, compute_input_statistics, standardize_input



In [4]:
training_experiment = 'ESD_pseudo_reality'
if training_experiment == 'ESD_pseudo_reality':
        period_training = '1961-1980'
        train_year_start=1961
        train_year_end=1980
        first_year=1961
input_path=f"/leonardo_work/ICT25_ESP/sdigioia/CORDEX-ML/CORDEX-graphs/ALPS_domain/{training_experiment}/"
graph_file = f"low_high_graph.pkl"
target_file=f"target.pkl"

In [14]:
# Load the graph and target files
with open(input_path+ graph_file, 'rb') as f:
        low_high_graph = pickle.load(f)

with open(input_path+ target_file, 'rb') as f:
        target_train = pickle.load(f)

target_train = target_train.T

In [15]:
torch.mean(target_train), torch.min(target_train), torch.max(target_train)

(tensor(3.0442), tensor(0.), tensor(339.8424))

In [ ]:
hist, bin_edges= np.histogram(target_train, bins=100)

In [6]:
target_train = prepare_target_Rall(target_train, threshold = 0.1)

In [7]:
type(target_train)

torch.Tensor

In [9]:
target_train.shape

torch.Size([15288, 7305])

In [12]:
torch.mean(target_train), torch.min(target_train), torch.max(target_train)

(tensor(0.7082), tensor(0.), tensor(5.8313))

In [13]:
torch.isnan(target_train).any()

tensor(False)

In [ ]:
def derive_qmse_bins_Rall(target_train, train_idxs, threshold=0.1):

    bins = np.arange(np.log1p(threshold), np.log1p(200), np.log1p(0.5))

    bins = np.insert(bins, 0, np.log1p(0))
    # consider only the time indices that are part of the training set
    values_unif_log, edges_unif_log = np.histogram(target_train[:,train_idxs].numpy(), bins=bins, density=False)
    # Assign bins to targets
    target_bins = np.digitize(target_train.numpy(), edges_unif_log, right=False).astype(float) - 1

    nbins = (np.nanmax(target_bins) + 1).astype(int)
    if nbins > len(values_unif_log):
        print(f"\nBins min: {np.nanmin(target_bins).astype(int)}, bins max: {np.nanmax(target_bins).astype(int)}, nbins: {nbins}, len weights: {len(values_unif_log)}")
        target_bins[target_bins == nbins -1] = nbins - 2
        nbins = nbins - 1

    print(f"\nbins min: {np.nanmin(target_bins).astype(int)}, bins max: {np.nanmax(target_bins).astype(int)}, nbins: {nbins}")
    target_bins = torch.tensor(target_bins)
    target_bins[torch.isnan(target_train)] = torch.nan

    return target_bins, values_unif_log, edges_unif_log

In [ ]:
target_bins, values_histo, edges_histo = derive_qmse_bins_Rall(target_train, train_idxs, threshold=0.1)